# 05 · IDEAM: detecciones tempranas de deforestación, por celda y trimestre

**Qué hace.** Baja los puntos de las detecciones tempranas de deforestación (DTD) que el IDEAM publica cada
trimestre y cuenta cuántos caen en cada celda de 5 km, y cuántos de ellos dentro de un área protegida del
SINAP.

**Cuándo correrlo.** Después de `02_carga_hansen`. Para actualizar, cada trimestre: los trimestres nuevos
aparecen solos; solo hay que agregar el año en la celda 1 cuando empieza uno nuevo.

**Qué lee.** Las tablas `GRILLA_CELDAS_5KM` y `BOSQUE_BASE_HANSEN_CELDA_5KM`, y los archivos KML de puntos del servidor público del
IDEAM (sin cuenta ni llave).

**Qué escribe**, en `DATASET_DEFORESTACION`:

| Tabla | Qué es |
|---|---|
| `DETECCIONES_TEMPRANAS_IDEAM_CELDA_5KM_TRIMESTRE` | Una fila por celda y trimestre: número de detecciones y cuántas en áreas protegidas |

**Qué tienes que cambiar.** Nada: la celda 1 ya apunta al proyecto del Observatorio (`prueba-ofr`).

**Cuánto tarda.** Unos cinco minutos.

**Cómo usar esta tabla, y cómo no.** Los puntos no traen superficie: la tabla cuenta detecciones, no
hectáreas. Y el IDEAM cambió a lo largo de los años con cuánto detalle divide sus detecciones, así que **los
conteos no se pueden comparar entre trimestres**: sumados por año se mueven en contra de la deforestación
real (correlación de −0,669 con la cifra oficial en 2021-2024). Sirve **dentro** de cada trimestre, para
decir qué lugares están peor que otros, con respaldo oficial. Para cuánto y para la tendencia están los
notebooks 03 y 04.

In [ ]:
# =====================================================================
# 1. CONFIGURACIÓN
#    Ya está llena para el proyecto del Observatorio (prueba-ofr). Solo
#    hay que cambiarla si las tablas se pasan a otro proyecto o dataset.
# =====================================================================
# --- Dónde están las tablas en BigQuery ------------------------------
# Proyecto de Google Cloud del Observatorio (Prueba-OFR). Es el id,
# no el nombre: el que aparece en la consola al elegir el proyecto.
PROYECTO = "prueba-ofr"

# Dataset con las tablas de deforestación que arman los notebooks 01
# a 06 (grilla, paneles y panel municipal).
DATASET_DEFORESTACION = "Cambio_Climatico"

# Ubicación de los datos. Debe ser la misma de las tablas del
# Observatorio (en BigQuery: abre una tabla > Detalles > Ubicación de
# los datos). Las del Observatorio están hoy en US.
UBICACION = "US"

# --- Para actualizar --------------------------------------------------
# Años que se buscan en el servidor del IDEAM. Los trimestres de cada año
# se leen del propio servidor, así que los nuevos aparecen solos. Cuando
# empiece un año nuevo, agrégalo a la lista.
DTD_ANIOS = [2020, 2021, 2022, 2023, 2024, 2025, 2026]

# --- Del trabajo de grado (no se cambia) -------------------------------
BOSQUE_MINIMO_HA = 50     # celdas con menos bosque base quedan fuera, como en los otros paneles

# --- Solo si corres el notebook en tu computador ----------------------
# En BigQuery déjalo vacío. En un computador: la ruta al archivo .json
# de la llave de una cuenta de servicio con acceso a estas tablas.
RUTA_LLAVE_BIGQUERY = ""

# Carpeta donde se bajan los archivos mientras corre el notebook.
# Vacío: una carpeta temporal. Si ya los tienes bajados, apúntala ahí
# y no se vuelven a descargar.
CARPETA_DESCARGAS = ""

In [ ]:
# =====================================================================
# 2. INSTALAR PAQUETES
#    En BigQuery instala lo que le falte a la máquina; la primera vez
#    tarda uno o dos minutos. No hay que cambiar nada. En un computador
#    con el entorno del repositorio (requirements_local.txt) se puede
#    saltar.
#    Las versiones fijas son las que se usaron en el trabajo de grado,
#    para que los resultados salgan iguales.
# =====================================================================
%pip install --quiet pyproj requests pandas-gbq

In [ ]:
# =====================================================================
# 3. CONEXIÓN CON BIGQUERY Y FUNCIONES DE APOYO
#    No hay que cambiar nada en esta celda. Se conecta a BigQuery y
#    define las funciones con que el resto del notebook lee y escribe
#    tablas. Es igual en todos los notebooks de esta carpeta.
# =====================================================================
import json
import re
import tempfile
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from google.cloud import bigquery

NOMBRE_NOTEBOOK = "05_carga_ideam_dtd"

# --- 3.1 ¿Quedó algo sin llenar en la celda 1? --------------------------
# Si queda algún <<LLENAR ...>>, el notebook se detiene aquí y dice cuál.
_pendientes = sorted(n for n, v in list(globals().items())
                     if n.isupper() and isinstance(v, str) and "<<LLENAR" in v)
if _pendientes:
    raise ValueError("Falta llenar en la celda 1: " + ", ".join(_pendientes))

# --- 3.2 Conexión ---------------------------------------------------------
# En BigQuery el notebook entra con la cuenta de quien lo corre y no
# necesita llave. La llave solo se usa si se corre en un computador.
if globals().get("RUTA_LLAVE_BIGQUERY"):
    from google.oauth2 import service_account
    client = bigquery.Client(
        project=PROYECTO,
        credentials=service_account.Credentials.from_service_account_file(RUTA_LLAVE_BIGQUERY))
else:
    client = bigquery.Client(project=PROYECTO)
print(f"Conectado a BigQuery: proyecto {PROYECTO}")

# Carpeta para los archivos que se descargan mientras corre el notebook.
# Es de paso: todo lo que importa termina en tablas de BigQuery.
_carpeta = globals().get("CARPETA_DESCARGAS") or ""
DIR_TRABAJO = Path(_carpeta) if _carpeta else Path(tempfile.gettempdir()) / "ofr_deforestacion"
DIR_TRABAJO.mkdir(parents=True, exist_ok=True)


# --- 3.3 Nombres de tabla -----------------------------------------------
def nombre_completo(tabla, dataset=None):
    """'deforestacion_ideam_celda_5km_periodo' -> 'proyecto.dataset.DEFORESTACION_IDEAM_CELDA_5KM_PERIODO'.

    Las tablas que crean estos notebooks se nombran en mayúscula
    (GRILLA_CELDAS_5KM, DEFORESTACION_IDEAM_CELDA_5KM_PERIODO...). Si la tabla ya viene completa
    (proyecto.dataset.tabla, como las del Observatorio), se devuelve igual.
    Si no, se arma con PROYECTO y el dataset indicado; por defecto,
    DATASET_DEFORESTACION.
    """
    if tabla.count(".") == 2:
        return tabla
    return f"{PROYECTO}.{dataset or globals().get('DATASET_DEFORESTACION')}.{tabla.upper()}"


# --- 3.4 Leer -------------------------------------------------------------
_MARCA = re.compile(r"^(\[índice(?: sin nombre)?\] )?(?:Nombre original: «(.*?)»)?")


_LECTURA_RAPIDA = [True]


def leer_sql(sql):
    """Corre una consulta SQL en BigQuery y devuelve un DataFrame de pandas.

    Usa la lectura rápida de BigQuery (Storage Read API). Si no está
    disponible (por ejemplo, porque la cuenta no tiene el rol «BigQuery
    Read Session User»), sigue con la lectura normal, que da el mismo
    resultado pero tarda más en tablas grandes.
    """
    trabajo = client.query(sql)
    trabajo.result()                      # si la consulta tiene un error, sale aquí
    resultado = None
    if _LECTURA_RAPIDA[0]:
        try:
            resultado = trabajo.to_arrow()
        except Exception as e:
            _LECTURA_RAPIDA[0] = False
            print("Aviso: la lectura rápida de BigQuery no está disponible "
                  f"({type(e).__name__}); se usa la lectura normal.")
    if resultado is None:
        resultado = trabajo.to_arrow(create_bqstorage_client=False)
    return resultado.to_pandas(date_as_object=False, coerce_temporal_nanoseconds=True)


def leer_tabla(tabla, columnas=None, dataset=None, filtro=None):
    """Lee una tabla de BigQuery a pandas.

    columnas  lista de columnas a traer; por defecto, todas.
    filtro    condición SQL opcional, p. ej. "anio >= 2021".

    Si la tabla se guardó con guardar_tabla(), devuelve las filas en el
    mismo orden en que se guardaron, las columnas con su nombre original
    (tildes, espacios) y el índice.
    """
    destino = nombre_completo(tabla, dataset)
    esquema = client.get_table(destino).schema
    con_orden = any(c.name == "orden_fila" for c in esquema)
    seleccion = ", ".join(f"`{c}`" for c in columnas) if columnas else "*"
    df = leer_sql(f"SELECT {seleccion} FROM `{destino}`"
                  + (f" WHERE {filtro}" if filtro else "")
                  + (" ORDER BY orden_fila" if con_orden else ""))
    df = df.drop(columns=["orden_fila"], errors="ignore")

    originales, indice, sin_nombre = {}, [], set()
    for campo in esquema:
        marca = _MARCA.match(campo.description or "")
        if marca.group(2):
            originales[campo.name] = marca.group(2)
        if marca.group(1) and campo.name in df.columns:
            indice.append(campo.name)
            if "sin nombre" in marca.group(1):
                sin_nombre.add(campo.name)
    if indice:
        df = df.set_index(indice)
        df.index = df.index.set_names(
            [None if n in sin_nombre else originales.get(n, n) for n in indice])
    return df.rename(columns=originales)


# --- 3.5 Escribir -----------------------------------------------------------
_DATASETS_LISTOS = set()


def _nombre_para_bigquery(nombre, usados):
    """Nombre de columna que BigQuery acepta siempre: letras sin tilde,
    números y guion bajo. Los que ya cumplen se dejan igual."""
    texto = str(nombre)
    if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,299}", texto) and texto.lower() not in usados:
        usados.add(texto.lower())
        return texto
    limpio = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    limpio = re.sub(r"[^0-9A-Za-z]+", "_", limpio).strip("_").lower() or "columna"
    if limpio[0].isdigit():
        limpio = "c_" + limpio
    limpio, base, k = limpio[:280], limpio[:280], 2
    while limpio.lower() in usados:
        limpio, k = f"{base}_{k}", k + 1
    usados.add(limpio.lower())
    return limpio


def _tipos_para_bigquery(df):
    """Deja cada columna en un tipo que BigQuery reciba sin error:
    categorías a texto, enteros con vacíos a entero, mezclas a texto."""
    for c in df.columns:
        s = df[c]
        if isinstance(s.dtype, pd.CategoricalDtype):
            s = s.astype(object).where(s.notna(), None)
        if s.dtype == object:
            valores = s.dropna()
            tipos = {type(v) for v in valores}
            if not tipos or tipos <= {str}:
                pass
            elif all(isinstance(v, (bool, np.bool_)) for v in valores):
                s = s.astype("boolean")
            elif all(isinstance(v, (int, np.integer)) and not isinstance(v, bool) for v in valores):
                s = s.astype("Int64")
            elif all(isinstance(v, (int, float, np.integer, np.floating)) for v in valores):
                s = s.astype(float)
            else:
                s = s.map(lambda v: None if v is None or v is pd.NA
                          or (isinstance(v, float) and v != v) else str(v))
        df[c] = s
    return df


def guardar_tabla(df, tabla, descripcion, dataset=None, indice=False,
                  descripciones=None):
    """Escribe `df` en BigQuery. Si la tabla ya existe, la REEMPLAZA entera.

    tabla          nombre corto, p. ej. "deforestacion_ideam_celda_5km_periodo".
    descripcion    texto que se ve en BigQuery, en los detalles de la tabla.
    dataset        por defecto, DATASET_DEFORESTACION. Si no existe, se crea.
    indice         True para guardar también el índice de pandas como
                   columna(s); leer_tabla() lo recupera.
    descripciones  {columna: texto}, para documentar columnas en BigQuery.
    """
    dataset = dataset or globals().get("DATASET_DEFORESTACION")
    destino = nombre_completo(tabla, dataset)
    if dataset not in _DATASETS_LISTOS:
        ds = bigquery.Dataset(f"{PROYECTO}.{dataset}")
        ds.location = globals().get("UBICACION", "US")
        client.create_dataset(ds, exists_ok=True)
        _DATASETS_LISTOS.add(dataset)

    datos = df.copy()
    marcas = {}
    if indice:
        nombres_indice = list(datos.index.names)
        datos = datos.reset_index()
        for k, n in enumerate(nombres_indice):
            columna = datos.columns[k]
            marcas[columna] = "[índice sin nombre] " if n is None else "[índice] "
    else:
        datos = datos.reset_index(drop=True)
    if isinstance(datos.columns, pd.MultiIndex):
        datos.columns = [" | ".join(str(x) for x in c if str(x) != "") for c in datos.columns]

    descripciones = descripciones or {}
    usados, notas, columnas = set(), {}, []
    for c in datos.columns:
        nuevo = _nombre_para_bigquery(c, usados)
        texto = marcas.get(c, "")
        if nuevo != str(c):
            texto += f"Nombre original: «{c}»"
        if descripciones.get(c):
            texto += (". " if texto and not texto.endswith("] ") else "") + descripciones[c]
        notas[nuevo] = texto[:1024]
        columnas.append(nuevo)
    datos.columns = columnas
    datos = _tipos_para_bigquery(datos)
    # BigQuery no guarda el orden de las filas, y algunos cálculos (el
    # muestreo del random forest, por ejemplo) dependen de él. Esta
    # columna lo conserva; leer_tabla() la usa y la quita.
    datos["orden_fila"] = np.arange(len(datos), dtype="int64")
    notas["orden_fila"] = ("Posición de la fila en la tabla original. Sirve para leerla "
                           "en el mismo orden: ORDER BY orden_fila.")

    trabajo = client.load_table_from_dataframe(
        datos, destino,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    trabajo.result()

    t = client.get_table(destino)
    t.description = descripcion
    t.schema = [bigquery.SchemaField(f.name, f.field_type, mode=f.mode,
                                     description=notas.get(f.name) or None,
                                     fields=f.fields) for f in t.schema]
    client.update_table(t, ["description", "schema"])
    print(f"Guardada {destino}: {t.num_rows:,} filas, {len(t.schema)} columnas")

## 4. La grilla y el bosque base

In [ ]:
LADO_CELDA_M = 5_000          # los de la grilla (notebook 01)
CRS_GRILLA = "EPSG:3116"

grilla = leer_tabla("grilla_celdas_5km")
bosque = leer_tabla("bosque_base_hansen_celda_5km")
if not grilla["cell_id"].equals(bosque["cell_id"]):
    raise RuntimeError("bosque_base_hansen_celda_5km no corresponde a esta grilla: vuelve a correr el notebook 02.")
print(f"Grilla: {len(grilla):,} celdas")

## 5. Qué trimestres hay, y descargarlos

El IDEAM publica cada trimestre un archivo KML de puntos en `<año>/Puntos/`. El nombre sigue el patrón
`atd_<año>_<trimestre en romano>_trim.kml`, pero con mayúsculas y minúsculas mezcladas, así que en vez de
adivinarlo se lee el índice de la carpeta en el servidor. Si un año no tiene carpeta de puntos, se avisa y
se sigue.

Igual que con las capas del notebook 04, la dirección tiene que ser `https`.

In [ ]:
import re

import requests

URL_DTD = "https://bart.ideam.gov.co/smbyc/Boletines%20Detecciones%20Tempranas%20de%20Deforestacion"
ROMANOS = {"i": 1, "ii": 2, "iii": 3, "iv": 4}

catalogo = {}
for anio in DTD_ANIOS:
    try:
        r = requests.get(f"{URL_DTD}/{anio}/Puntos/", timeout=60)
    except Exception as e:
        print(f"  {anio}: no se pudo leer la carpeta ({e})")
        continue
    if r.status_code != 200:
        print(f"  {anio}: sin carpeta de puntos (HTTP {r.status_code})")
        continue
    for nombre in re.findall(r'href="([^"?/]+\.kml)"', r.text, re.I):
        m = re.search(rf"atd_{anio}_(i{{1,3}}|iv)_trim", nombre, re.I)
        if m:
            catalogo[(anio, ROMANOS[m.group(1).lower()])] = nombre
catalogo = dict(sorted(catalogo.items()))
if not catalogo:
    raise RuntimeError("El servidor del IDEAM no devolvió ningún trimestre.")
print(f"{len(catalogo)} trimestres publicados: {catalogo[min(catalogo)]} a {catalogo[max(catalogo)]}")


def descargar_trimestre(anio, trimestre, nombre, intentos=3):
    """Baja el KML de un trimestre, salvo que ya esté. Devuelve su ruta."""
    destino = DIR_TRABAJO / f"atd_{anio}_{trimestre}.kml"
    if destino.exists() and destino.stat().st_size > 0:
        return destino
    temporal = destino.with_suffix(".parcial")
    for intento in range(1, intentos + 1):
        try:
            with requests.get(f"{URL_DTD}/{anio}/Puntos/{nombre}", stream=True, timeout=300) as r:
                r.raise_for_status()
                total = int(r.headers.get("Content-Length", 0))
                bajado = 0
                with open(temporal, "wb") as f:
                    for trozo in r.iter_content(chunk_size=1 << 20):
                        f.write(trozo)
                        bajado += len(trozo)
            if total and bajado != total:
                raise IOError(f"llegó incompleto: {bajado} de {total} bytes")
            temporal.replace(destino)
            return destino
        except Exception as e:
            print(f"  {anio}-T{trimestre}: intento {intento} de {intentos} falló ({e})")
            temporal.unlink(missing_ok=True)
    raise RuntimeError(f"No se pudo bajar {anio}-T{trimestre}. Vuelve a correr esta celda.")


ARCHIVOS = {(a, t): descargar_trimestre(a, t, n) for (a, t), n in catalogo.items()}
print(f"{len(ARCHIVOS)} trimestres en la carpeta de trabajo")

## 6. Leer los puntos

Cada KML trae un punto por detección, con atributos: municipio, vereda, corporación autónoma regional y,
en algunos trimestres, el área protegida del SINAP donde cae (`NOM_SINAP`). Se leen con el lector de XML de
Python, sin depender de librerías geográficas. Se comprobó que da los mismos puntos y atributos que
`geopandas.read_file` en los 25 trimestres de 2020 a 2026.

El campo `NOM_SINAP` no sigue una sola convención: en algunos trimestres no existe; en otros, los puntos
fuera de un área protegida dicen "No Aplica" o "NO APLICA", y en otros vienen vacíos. Por eso se normaliza
el texto antes de decidir. Si un trimestre no trae el campo, su conteo de SINAP queda vacío y no en cero:
no tener el dato no es lo mismo que no tener detecciones en áreas protegidas.

In [ ]:
import xml.etree.ElementTree as ET


def leer_kml_puntos(ruta):
    """Puntos de un KML como DataFrame: lon, lat y los campos de sus atributos."""
    raiz = ET.parse(ruta).getroot()
    nombre_local = lambda e: e.tag.rsplit("}", 1)[-1]          # sin el espacio de nombres del XML
    campos = [e.get("name") for e in raiz.iter() if nombre_local(e) == "SimpleField"]
    filas = []
    for marca in (e for e in raiz.iter() if nombre_local(e) == "Placemark"):
        coordenadas = next((e.text for e in marca.iter() if nombre_local(e) == "coordinates"), None)
        if not coordenadas:
            continue
        lon, lat = (float(v) for v in coordenadas.strip().split(",")[:2])
        fila = {"lon": lon, "lat": lat}
        fila.update({e.get("name"): e.text for e in marca.iter() if nombre_local(e) == "SimpleData"})
        filas.append(fila)
    puntos = pd.DataFrame(filas)
    for c in campos:                       # un campo del esquema sin ningún valor queda vacío
        if c not in puntos.columns:
            puntos[c] = None
    return puntos


def en_sinap(puntos):
    """(marca de cada punto dentro de un área protegida, ¿el trimestre trae el dato?)."""
    if "NOM_SINAP" not in puntos.columns:
        return np.zeros(len(puntos), dtype=bool), False
    texto = (puntos["NOM_SINAP"].astype("string")
             .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")
             .str.strip().str.lower())
    dentro = texto.notna() & (texto != "") & (~texto.str.startswith("no aplica", na=False))
    return dentro.to_numpy(dtype=bool), True

## 7. Detecciones por celda y trimestre

Los puntos vienen en grados (EPSG:4326). Se pasan a la proyección de la grilla y la celda de cada punto es
`floor(x / 5000)`, la misma regla que usan los notebooks 02 y 04 para los píxeles. Así un punto y un píxel
del mismo lugar terminan en la misma celda.

In [ ]:
from pyproj import Transformer

a_grilla = Transformer.from_crs("EPSG:4326", CRS_GRILLA, always_xy=True)
llave = {(int(r.ix), int(r.iy)): r.cell_id for r in grilla.itertuples()}
conteos = []
for (anio, trimestre), ruta in ARCHIVOS.items():
    puntos = leer_kml_puntos(ruta)
    if puntos.empty:
        print(f"  {anio}-T{trimestre}: sin registros")
        continue
    x, y = a_grilla.transform(puntos["lon"].to_numpy(), puntos["lat"].to_numpy())
    ix = np.floor(np.asarray(x) / LADO_CELDA_M).astype(np.int64)
    iy = np.floor(np.asarray(y) / LADO_CELDA_M).astype(np.int64)
    celda = [llave.get((a, b)) for a, b in zip(ix, iy)]
    sinap, hay_sinap = en_sinap(puntos)

    d = pd.DataFrame({"cell_id": celda, "en_sinap": sinap}).dropna(subset=["cell_id"])
    agregado = (d.groupby("cell_id").agg(n_alertas=("en_sinap", "size"), n_alertas_sinap=("en_sinap", "sum"))
                 .reset_index())
    agregado["anio"], agregado["trimestre"] = anio, trimestre
    agregado["sinap_disponible"] = hay_sinap
    conteos.append(agregado)
    fuera = len(puntos) - len(d)
    print(f"  {anio}-T{trimestre}: {len(puntos):6,} detecciones en {len(agregado):5,} celdas · SINAP: "
          + (f"{int(sinap.sum()):,} ({100 * sinap.mean():.1f} %)" if hay_sinap else "sin dato")
          + (f" · {fuera} fuera de la grilla" if fuera else ""))
conteo = pd.concat(conteos, ignore_index=True)

## 8. La tabla

Una fila por celda y trimestre, también los trimestres sin detecciones (con 0), igual que el panel de
alertas de GFW. `periodo` es el primer día del trimestre, para ordenar y cruzar con los otros paneles.
Se quedan las celdas con al menos 50 ha de bosque base, como en las otras tablas.

In [ ]:
MES_TRIMESTRE = {1: 1, 2: 4, 3: 7, 4: 10}
periodos = conteo[["anio", "trimestre"]].drop_duplicates().sort_values(["anio", "trimestre"])
indice = pd.MultiIndex.from_product([grilla["cell_id"].values, [tuple(x) for x in periodos.to_numpy()]],
                                    names=["cell_id", "_per"])
panel_dtd = pd.DataFrame(index=indice).reset_index()
panel_dtd[["anio", "trimestre"]] = pd.DataFrame(panel_dtd["_per"].tolist(), index=panel_dtd.index)
panel_dtd = panel_dtd.drop(columns="_per")

panel_dtd = panel_dtd.merge(conteo, on=["cell_id", "anio", "trimestre"], how="left")
panel_dtd[["n_alertas", "n_alertas_sinap"]] = panel_dtd[["n_alertas", "n_alertas_sinap"]].fillna(0).astype(int)
# Si el trimestre trae el dato del SINAP lo trae para todas sus celdas, también las que no tienen detecciones.
disponible = conteo.groupby(["anio", "trimestre"])["sinap_disponible"].first().rename("disp")
panel_dtd = panel_dtd.merge(disponible, on=["anio", "trimestre"], how="left")
panel_dtd["sinap_disponible"] = panel_dtd.pop("disp").fillna(False).astype(bool)
panel_dtd.loc[~panel_dtd.sinap_disponible, "n_alertas_sinap"] = pd.NA

panel_dtd["periodo"] = pd.to_datetime(dict(year=panel_dtd.anio, month=panel_dtd.trimestre.map(MES_TRIMESTRE), day=1))
panel_dtd["trimestre_txt"] = panel_dtd.anio.astype(str) + "-T" + panel_dtd.trimestre.astype(str)

meta = grilla[["cell_id", "lon", "lat", "departamento"]].copy()
meta["bosque_base_ha"] = bosque["bosque_base_ha"].values
panel_dtd = panel_dtd.merge(meta, on="cell_id", how="left")
panel_dtd = panel_dtd[panel_dtd["bosque_base_ha"] >= BOSQUE_MINIMO_HA].copy()
panel_dtd = panel_dtd.merge(grilla[["cell_id", "cod_dane", "municipio"]], on="cell_id", how="left")
columnas = ["cell_id", "periodo", "trimestre_txt", "anio", "trimestre", "n_alertas", "n_alertas_sinap",
            "sinap_disponible", "lon", "lat", "departamento", "bosque_base_ha", "cod_dane", "municipio"]
panel_dtd = panel_dtd[columnas].sort_values(["cell_id", "periodo"]).reset_index(drop=True)

con = panel_dtd[panel_dtd.sinap_disponible]
print(f"detecciones_tempranas_ideam_celda_5km_trimestre: {len(panel_dtd):,} filas = {panel_dtd.cell_id.nunique():,} celdas x "
      f"{panel_dtd.trimestre_txt.nunique()} trimestres")
print(f"Detecciones en áreas protegidas: {int(con.n_alertas_sinap.sum()):,} de {con.n_alertas.sum():,} "
      f"({100 * con.n_alertas_sinap.sum() / max(con.n_alertas.sum(), 1):.1f} %), en los "
      f"{con.trimestre_txt.nunique()} trimestres que traen el dato")
print("Detecciones por año (no comparables entre años, ver arriba):")
print(panel_dtd.groupby("anio")["n_alertas"].sum().to_string())

## 9. Guardar en BigQuery

In [ ]:
guardar_tabla(
    panel_dtd, "detecciones_tempranas_ideam_celda_5km_trimestre",
    "Detecciones tempranas de deforestación del IDEAM (SMByC) por celda de 5 km y trimestre. Cuenta puntos, "
    "no hectáreas, y los conteos NO son comparables entre trimestres: sirve para priorizar lugares dentro de "
    f"un mismo trimestre. Solo celdas con al menos {BOSQUE_MINIMO_HA} ha de bosque base.",
    descripciones={
        "cell_id": "Identificador de la celda (tabla GRILLA_CELDAS_5KM).",
        "periodo": "Primer día del trimestre.",
        "trimestre_txt": "Trimestre en texto, p. ej. 2024-T3.",
        "anio": "Año.",
        "trimestre": "Trimestre (1 a 4).",
        "n_alertas": "Número de detecciones tempranas que caen en la celda en el trimestre.",
        "n_alertas_sinap": "De ellas, cuántas caen en un área protegida del SINAP. Vacío si el trimestre no "
                           "trae ese dato.",
        "sinap_disponible": "Si el trimestre trae el dato del área protegida (campo NOM_SINAP).",
        "lon": "Longitud del centro de la celda.",
        "lat": "Latitud del centro de la celda.",
        "departamento": "Departamento del centro de la celda.",
        "bosque_base_ha": "Bosque según Hansen (tabla BOSQUE_BASE_HANSEN_CELDA_5KM).",
        "cod_dane": "Código DANE del municipio del centro de la celda.",
        "municipio": "Nombre del municipio, solo para leer.",
    })

Listo. Con los notebooks 01 a 05 quedan las cuatro tablas de deforestación. Sigue `06_carga_panel_municipal`,
que las cruza con las tablas del Observatorio.